# RadarMD — full-scale training on Colab

Drives full training on the complete NIH ChestX-ray14 dataset (112k images).
All real logic lives in `src/radarmd`; this notebook only clones, installs,
stages data, and launches runs.

**Runtime → Change runtime type → GPU (A100 or T4, High-RAM).**

Pipeline: download full dataset → pack resized JPEG shards to Drive **once** →
on every run, copy shards to the local SSD and extract → train.

## 1. Clone + install

In [ ]:
!git clone https://github.com/aarushnarang02/radarmd.git
%cd radarmd
# Colab already ships a CUDA torch; install the rest of the project + extras.
!pip install -q -e '.[dev,data,track,interpret]'

## 2. Credentials (Kaggle + Weights & Biases)

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
from google.colab import drive
drive.mount('/content/drive')

## 3. One-time: download full dataset and pack shards to Drive

Run this cell **once ever**. It writes ~4-5GB of 320px JPEG shards to Drive;
later sessions skip straight to step 4. (~30-45 min the first time.)

In [ ]:
SHARDS = '/content/drive/MyDrive/radarmd/shards'
if not os.path.exists(SHARDS) or not os.listdir(SHARDS):
    !python scripts/prepare_data.py --dest /content/data --full
    !python scripts/pack_shards.py --src /content/data --dest {SHARDS} --size 320 --shard-size 4096
    # Copy the small metadata CSVs to Drive next to the shards.
    !find /content/data -name '*.csv' -exec cp {{}} {SHARDS}/.. \;
else:
    print('Shards already present on Drive; skipping download/pack.')

## 4. Stage shards onto the local SSD and extract

In [ ]:
!mkdir -p /content/shards && cp {SHARDS}/*.tar /content/shards/
import radarmd.data.shards as S
n = S.extract_shards('/content/shards', '/content/images')
print(f'Extracted {n} images to /content/images')
# Metadata CSV staged from Drive.
META = '/content/drive/MyDrive/radarmd/Data_Entry_2017.csv'

## 5a. Baseline (for the '6-point gain' comparison)

Frozen-features / low-res ResNet-50 baseline. Its mean AUROC is the number the
tuned DenseNet is measured against.

In [ ]:
!python scripts/train.py --config configs/base.yaml \
    model.backbone=resnet50 data.image_size=224 optim.max_epochs=10 \
    data.data_dir=/content/images data.metadata_csv=$META \
    wandb.mode=online wandb.project=radarmd

## 5b. Tuned DenseNet-121 (headline run)

In [ ]:
!python scripts/train.py --config configs/densenet121.yaml \
    data.data_dir=/content/images data.metadata_csv=$META \
    wandb.mode=online wandb.project=radarmd

## 6. Hyperparameter sweep (the 60+ experiments)

Launch a Bayesian sweep; run one or more agents. Point `data.*` at the local
images via the sweep command or per-agent env.

In [ ]:
!wandb sweep configs/sweep.yaml
# then, with the printed sweep id:
# !wandb agent <entity>/radarmd/<sweep_id>